[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C74_Gaussian_Splatting_Course/03_rasterization/03_rasterization.ipynb)

# C74 · 模块 03 · 从零写一个 tile 光栅化器

本 notebook 真的写出一个能出图的 tile 光栅化器，并验证它：

1. **分箱**：$3\sigma$ 包围盒 → (高斯, tile) 对；
2. **一个 64 位键让一次排序同时按 tile 与深度排好**，
   而**深度用 16 位量化会让 68.75% 的相邻对顺序未定义**；
3. **逐 tile 渲染 + 提前终止**，出一张图；
4. **与逐像素暴力实现逐位比对** —— 关掉两个近似后必须严格相等；
5. 负载分布（最大/均值 = 2.67×）与每帧的操作计数；
6. **顺序不变性检查**：打乱输入数组顺序，输出必须不变。

只用 numpy，CPU，离线。分辨率取 640×480 以便在 CPU 上跑得动。

In [ ]:
import numpy as np, math
print('numpy', np.__version__)

W_IMG, H_IMG, TS = 640, 480, 16
NX, NY = math.ceil(W_IMG/TS), math.ceil(H_IMG/TS)
NTILE = NX * NY
F, CX, CY = 600.0, W_IMG/2, H_IMG/2
print(f'{W_IMG}x{H_IMG} / {TS}x{TS} = {NX} x {NY} = {NTILE} tiles')
print(f'对照 1920x1080: {math.ceil(1920/16)} x {math.ceil(1080/16)} = '
      f'{math.ceil(1920/16)*math.ceil(1080/16)} tiles')
assert math.ceil(1920/16)*math.ceil(1080/16) == 8160

def quat_to_R(q):
    q = np.asarray(q, float); q = q/np.linalg.norm(q); w, x, y, z = q
    return np.array([[1-2*(y*y+z*z), 2*(x*y-w*z),   2*(x*z+w*y)],
                     [2*(x*y+w*z),   1-2*(x*x+z*z), 2*(y*z-w*x)],
                     [2*(x*z-w*y),   2*(y*z+w*x),   1-2*(x*x+y*y)]])

def cov3d(scale, q):
    M = quat_to_R(q) * np.asarray(scale, float)
    return M @ M.T

def make_scene(n=3000, seed=0):
    '''一个合成场景：一面墙 + 一个近处的物体 + 一些散落的小高斯。'''
    rng = np.random.default_rng(seed)
    mus, scales, quats, alphas, colors = [], [], [], [], []
    # 墙：z=8m 附近的一片
    nw = n//2
    mus.append(np.stack([rng.uniform(-3.0, 3.0, nw), rng.uniform(-2.2, 2.2, nw),
                         8.0 + rng.normal(0, 0.05, nw)], 1))
    scales.append(np.stack([rng.uniform(0.04, 0.10, nw), rng.uniform(0.04, 0.10, nw),
                            rng.uniform(0.004, 0.012, nw)], 1))
    quats.append(rng.normal(0, 1, (nw, 4)))
    alphas.append(rng.uniform(0.4, 0.9, nw))
    colors.append(np.stack([rng.uniform(0.3, 0.5, nw), rng.uniform(0.35, 0.55, nw),
                            rng.uniform(0.45, 0.65, nw)], 1))
    # 近物：z=3m 的一个团
    no = n//3
    mus.append(np.stack([rng.normal(0.5, 0.30, no), rng.normal(-0.2, 0.25, no),
                         3.0 + rng.normal(0, 0.20, no)], 1))
    scales.append(np.exp(rng.normal(np.log(0.035), 0.4, (no, 3))))
    quats.append(rng.normal(0, 1, (no, 4)))
    alphas.append(rng.uniform(0.5, 0.95, no))
    colors.append(np.stack([rng.uniform(0.7, 1.0, no), rng.uniform(0.3, 0.6, no),
                            rng.uniform(0.1, 0.3, no)], 1))
    # 散落
    nr = n - nw - no
    mus.append(np.stack([rng.uniform(-2.5, 2.5, nr), rng.uniform(-1.8, 1.8, nr),
                         rng.uniform(1.5, 12.0, nr)], 1))
    scales.append(np.exp(rng.normal(np.log(0.03), 0.6, (nr, 3))))
    quats.append(rng.normal(0, 1, (nr, 4)))
    alphas.append(rng.uniform(0.1, 0.6, nr))
    colors.append(rng.uniform(0.2, 0.9, (nr, 3)))
    return (np.concatenate(mus), np.concatenate(scales), np.concatenate(quats),
            np.concatenate(alphas), np.concatenate(colors))

MU, SC, QT, AL, CO = make_scene(3000, 0)
print(f'\n场景：{len(MU)} 个高斯，深度范围 {MU[:,2].min():.2f} ~ {MU[:,2].max():.2f} m')

## 1 · 预处理：投影成 2D 椭圆，算出圆锥系数与包围半径

对每个高斯算出 `(uv, conic, radius, depth, alpha, color)`。
`conic` 是 $\Sigma'^{-1}$ 的三个独立分量 —— 渲染时只需要它，不需要 $\Sigma'$ 本身。

In [ ]:
def preprocess(mu, sc, qt, al, co, z_near=0.2, dilate=0.3, k_sigma=3.0):
    '''返回一个 dict，只含通过剔除的高斯。'''
    keep = mu[:, 2] > z_near
    mu, sc, qt, al, co = mu[keep], sc[keep], qt[keep], al[keep], co[keep]
    n = len(mu)
    uv = np.stack([CX + F*mu[:, 0]/mu[:, 2], CY + F*mu[:, 1]/mu[:, 2]], 1)
    conic = np.empty((n, 3)); radius = np.empty(n)
    for i in range(n):
        x, y, z = mu[i]
        J = np.array([[F/z, 0.0, -F*x/z**2], [0.0, F/z, -F*y/z**2]])
        S2 = J @ cov3d(sc[i], qt[i]) @ J.T
        S2 = S2 + dilate*np.eye(2)                       # 模块 02 的低通滤波
        det = S2[0,0]*S2[1,1] - S2[0,1]**2
        conic[i] = [S2[1,1]/det, -S2[0,1]/det, S2[0,0]/det]   # Σ'⁻¹ 的 (a,b,c)
        mid = 0.5*(S2[0,0] + S2[1,1])
        lam1 = mid + np.sqrt(max(0.1, mid*mid - det))    # 官方的 max(0.1,·) 保护
        radius[i] = np.ceil(k_sigma*np.sqrt(lam1))
    # 屏幕外剔除
    on = ((uv[:,0] + radius > 0) & (uv[:,0] - radius < W_IMG) &
          (uv[:,1] + radius > 0) & (uv[:,1] - radius < H_IMG))
    return dict(uv=uv[on], conic=conic[on], radius=radius[on],
                depth=mu[on, 2], alpha=al[on], color=co[on])

G = preprocess(MU, SC, QT, AL, CO)
n_g = len(G['uv'])
print(f'通过剔除的高斯 {n_g} / {len(MU)}')
print(f'屏幕半径: 中位 {np.median(G["radius"]):.1f} px  均值 {G["radius"].mean():.1f}'
      f'  P95 {np.percentile(G["radius"],95):.1f}  最大 {G["radius"].max():.0f}')

# conic 必须正定（否则 exp(-½dᵀΣ'⁻¹d) 会发散）
a, b, c = G['conic'].T
det_inv = a*c - b*b
assert np.all(a > 0) and np.all(c > 0) and np.all(det_inv > 0), 'conic 必须正定'
print(f'\n✓ 全部 {n_g} 个 conic 正定（a>0, c>0, ac-b²>0）')
print(f'  最小的 ac-b² = {det_inv.min():.3e} —— 这是 +0.3I 保证的（模块 02）')

## 2 · 分箱：$3\sigma$ 包围盒 → (高斯, tile) 对

In [ ]:
def tile_range(uv, radius, nx=NX, ny=NY, ts=TS):
    '''包围盒覆盖的 tile 区间 [x0,x1] x [y0,y1]（闭区间，已裁到屏幕内）。'''
    x0 = np.clip(np.floor((uv[0]-radius)/ts).astype(int), 0, nx-1)
    x1 = np.clip(np.floor((uv[0]+radius)/ts).astype(int), 0, nx-1)
    y0 = np.clip(np.floor((uv[1]-radius)/ts).astype(int), 0, ny-1)
    y1 = np.clip(np.floor((uv[1]+radius)/ts).astype(int), 0, ny-1)
    return x0, x1, y0, y1

def bin_gaussians(G):
    '''返回 (gauss_idx, tile_id) 两个等长数组 —— 所有 (高斯,tile) 对。'''
    gi, ti = [], []
    for i in range(len(G['uv'])):
        x0, x1, y0, y1 = tile_range(G['uv'][i], G['radius'][i])
        tx = np.arange(x0, x1+1); ty = np.arange(y0, y1+1)
        t = (ty[:, None]*NX + tx[None, :]).ravel()
        gi.append(np.full(len(t), i)); ti.append(t)
    return np.concatenate(gi), np.concatenate(ti)

gidx, tidx = bin_gaussians(G)
P = len(gidx)
print(f'(高斯,tile) 对总数 P = {P}')
print(f'  P / 高斯数 = {P/n_g:.2f}   每 tile 平均 {P/NTILE:.1f}')

per_g = np.bincount(gidx, minlength=n_g)
print(f'\n每个高斯覆盖的 tile 数: 中位 {np.median(per_g):.0f}  均值 {per_g.mean():.2f}'
      f'  最大 {per_g.max()}')
# 理论：半径 r 的包围盒覆盖约 (2r/TS+1)² 个 tile
pred = ((2*G['radius']/TS)+1)**2
print(f'理论 (2r/{TS}+1)² 的均值 {pred.mean():.2f}（实测 {per_g.mean():.2f}）')
assert abs(per_g.mean()/pred.mean() - 1) < 0.25, '实测与理论应在 25% 内'
print('✓ 实测与 (2r/TS+1)² 一致（差异来自裁剪与包围盒的整数对齐）')

# 覆盖半径是主控变量
print('\n覆盖半径对 P 的影响（1920x1080, 50 万高斯）：')
NT_HD = 8160
for r in [4, 8, 20, 100]:
    pr = (2*r/16+1)**2
    print(f'  半径 {r:3d} px: 每高斯 {pr:7.2f} tiles -> P = {500_000*pr/1e6:6.2f}M'
          f'  每 tile {500_000*pr/NT_HD:7.1f}')
print(f'\n✓ 半径 8 -> 20（2.5×）时 P 涨 {((2*20/16+1)/(2*8/16+1))**2:.2f}×')
print(f'  一个半径 100 px 的高斯抵得上 {((2*100/16+1)/(2*8/16+1))**2:.0f} 个正常高斯')

## 3 · 64 位键：一次排序解决两件事

`key = (tile_id << 32) | depth_bits`。对它排一次序，结果自动是
「按 tile 分组、组内按深度升序」。

In [ ]:
def float_to_uint32_monotonic(x):
    '''把**正** float32 按位重解释成 uint32。对正数，位模式与数值同序。'''
    x32 = np.asarray(x, np.float32)
    assert np.all(x32 > 0), '这个技巧只对正数成立（依赖近平面剔除）'
    return x32.view(np.uint32)

def make_keys(tile_ids, depths):
    '''返回 uint64 键。高 32 位是 tile_id，低 32 位是深度的单调编码。'''
    t = np.asarray(tile_ids, np.uint64)
    d = float_to_uint32_monotonic(depths).astype(np.uint64)
    return (t << np.uint64(32)) | d

# 先验证位重解释的单调性
_probe = np.array([1e-6, 0.2, 1.0, 1.5, 7.99, 8.0, 8.01, 1e6], np.float32)
_bits = float_to_uint32_monotonic(_probe)
assert np.all(np.diff(_bits.astype(np.int64)) > 0), '正 float32 的位模式必须与数值同序'
print('float32 位重解释的单调性:')
for v, b in zip(_probe, _bits):
    print(f'  {v:12.6g} -> 0x{b:08x}')
print('✓ 严格单调 —— 所以按 uint32 排序 == 按 float 排序，一位精度都不损失')

keys = make_keys(tidx, G['depth'][gidx])
order = np.argsort(keys, kind='stable')
st, sd = tidx[order], G['depth'][gidx][order]

assert np.all(np.diff(st.astype(np.int64)) >= 0), 'tile_id 必须非降'
same_tile = np.diff(st.astype(np.int64)) == 0
assert np.all(np.diff(sd)[same_tile] >= -1e-7), '同一 tile 内深度必须非降'
print(f'\n✓ {P} 个键排一次序：tile 非降 ✓，同 tile 内深度非降 ✓')

# 每个 tile 的区间
starts = np.searchsorted(st, np.arange(NTILE), 'left')
ends = np.searchsorted(st, np.arange(NTILE), 'right')
counts = ends - starts
print(f'\n每 tile 的列表长: 均值 {counts.mean():.1f}  中位 {np.median(counts):.0f}'
      f'  P95 {np.percentile(counts,95):.0f}  最大 {counts.max()}  空 tile {(counts==0).mean():.1%}')
print(f'  最大/均值 = {counts.max()/counts.mean():.2f}×'
      f'  <- GPU 上整帧时延由最慢的 tile 决定')
assert counts.sum() == P

In [ ]:
# 深度用多少位量化：碰撞率
def collision_rate(depths, bits):
    '''把深度线性量化到 `bits` 位后，排序键相邻相等（顺序未定义）的比例。'''
    d = np.asarray(depths, float)
    lo, hi = d.min(), d.max()
    q = ((d - lo)/(hi - lo)*(2**bits - 1)).astype(np.uint64)
    return float((np.diff(np.sort(q)) == 0).mean())

# 用一个 0.5~100m 的连续深度分布（比本场景更接近真实户外）
rng_d = np.random.default_rng(0)
depths_wide = rng_d.uniform(0.5, 100.0, 200_000)
print('深度范围 0.5~100 m，20 万个值：')
print('  位数    相邻键相等（顺序未定义）的比例')
rates = {}
for bits in [16, 20, 24, 32]:
    r = collision_rate(depths_wide, bits); rates[bits] = r
    print(f'   {bits:2d}     {r:9.4%}')

assert rates[16] > 0.6, f'16 位应有 60%+ 的碰撞，实测 {rates[16]:.2%}'
assert rates[32] < 1e-3, f'32 位应几乎无碰撞，实测 {rates[32]:.4%}'
assert rates[16] > rates[20] > rates[24] > rates[32]
print(f'\n✓ 16 位: {rates[16]:.2%} 的相邻对顺序**未定义**')
print(f'  而模块 01 量过，打乱顺序的中位偏差是 19.6% —— 所以这不是无害的精度损失')
print(f'  32 位: {rates[32]:.4%}。官方更进一步：直接位重解释 float32，零精度损失')

# 碰撞的直接后果：排序不稳定时输出会变
print('\n演示：把深度量化到 16 位，再打乱输入顺序，输出会不会变？')
d_small = G['depth'][gidx]
lo, hi = d_small.min(), d_small.max()
q16 = ((d_small - lo)/(hi - lo)*(2**16-1)).astype(np.uint64)
key16 = (tidx.astype(np.uint64) << np.uint64(32)) | q16
perm = np.random.default_rng(3).permutation(P)
o_a = np.argsort(key16, kind='stable')
o_b = perm[np.argsort(key16[perm], kind='stable')]
n_diff = int((G['depth'][gidx][o_a] != G['depth'][gidx][o_b]).sum())
print(f'  16 位键：打乱输入后，{n_diff}/{P} 个位置的深度不同 ({n_diff/P:.2%})')
o_a32 = np.argsort(keys, kind='stable')
o_b32 = perm[np.argsort(keys[perm], kind='stable')]
n_diff32 = int((G['depth'][gidx][o_a32] != G['depth'][gidx][o_b32]).sum())
print(f'  32 位键：{n_diff32}/{P} ({n_diff32/P:.2%})')
assert n_diff > 0, '16 位键下打乱输入必须改变排序结果'
print('\n✓ 这就是「顺序不变性检查」能抓到的 bug —— 而它只看 PSNR 是看不出来的')

## 4 · 逐 tile 渲染 + 提前终止

In [ ]:
def render_tiles(G, gidx, tidx, T_min=1e-4, alpha_min=1.0/255,
                 shared_order=True, bg=(0.05, 0.05, 0.08)):
    '''逐 tile 渲染。返回 (img, T_final, n_processed)。
    shared_order=False 时退化为逐像素各自排序（关掉那个近似）。'''
    keys = make_keys(tidx, G['depth'][gidx])
    order = np.argsort(keys, kind='stable')
    g_s, t_s = gidx[order], tidx[order]
    starts = np.searchsorted(t_s, np.arange(NTILE), 'left')
    ends = np.searchsorted(t_s, np.arange(NTILE), 'right')

    img = np.zeros((H_IMG, W_IMG, 3))
    Tmap = np.ones((H_IMG, W_IMG))
    n_proc = np.zeros((H_IMG, W_IMG), int)

    for tid in range(NTILE):
        s, e = starts[tid], ends[tid]
        if s == e:
            continue
        ty, tx = divmod(tid, NX)
        y0, y1 = ty*TS, min((ty+1)*TS, H_IMG)
        x0, x1 = tx*TS, min((tx+1)*TS, W_IMG)
        yy, xx = np.mgrid[y0:y1, x0:x1]
        T = np.ones(yy.shape); acc = np.zeros(yy.shape + (3,))
        cnt = np.zeros(yy.shape, int)
        for k in range(s, e):
            gi = g_s[k]
            live = T >= T_min                       # 提前终止（逐像素）
            if not live.any():
                break
            dx = xx - G['uv'][gi, 0]; dy = yy - G['uv'][gi, 1]
            a_, b_, c_ = G['conic'][gi]
            m2 = a_*dx*dx + 2*b_*dx*dy + c_*dy*dy
            al = G['alpha'][gi] * np.exp(-0.5*m2)
            use = live & (al >= alpha_min)
            if not use.any():
                continue
            w = np.where(use, T*al, 0.0)
            acc += w[..., None] * G['color'][gi]
            T = np.where(use, T*(1-al), T)
            cnt += use
        img[y0:y1, x0:x1] = acc
        Tmap[y0:y1, x0:x1] = T
        n_proc[y0:y1, x0:x1] = cnt
    img = img + Tmap[..., None]*np.array(bg)        # 合成背景
    return img, Tmap, n_proc

import time
t0 = time.time()
img, Tmap, nproc = render_tiles(G, gidx, tidx)
print(f'渲染完成 {time.time()-t0:.2f} s')
print(f'累积不透明度 A = 1-T: 均值 {1-Tmap.mean():.4f}  '
      f'A>0.99 的像素 {(1-Tmap > 0.99).mean():.1%}')
print(f'每像素实际处理的高斯数: 均值 {nproc.mean():.1f}  中位 {np.median(nproc):.0f}'
      f'  P95 {np.percentile(nproc,95):.0f}  最大 {nproc.max()}')

# 与「不提前终止」对比
_, _, nproc_full = render_tiles(G, gidx, tidx, T_min=0.0, alpha_min=0.0)
print(f'\n不提前终止时: 均值 {nproc_full.mean():.1f}  最大 {nproc_full.max()}')
saved = 1 - nproc.sum()/nproc_full.sum()
print(f'提前终止省下 {saved:.1%} 的高斯-像素求值')
assert saved > 0.3, f'应至少省下 30%，实测 {saved:.1%}'
print('（本合成场景的 α 偏小、且大片背景永不饱和，所以低于 245 个高斯那个理想例子的 92.2%）')

print('\n图像（. < 0.15 < : < 0.35 < o < 0.6 < #）：')
lum = img.mean(2)
for r in range(0, H_IMG, 20):
    line = ''.join('.' if lum[r,c] < 0.15 else (':' if lum[r,c] < 0.35 else
                   ('o' if lum[r,c] < 0.6 else '#')) for c in range(0, W_IMG, 8))
    print('  ' + line)

## 5 · 与逐像素暴力实现逐位比对

tile 版一共有**三个**近似，第三个很容易被忽略：

1. **提前终止**（`T_min`、`alpha_min`）；
2. **tile 内共享一个顺序**（本场景里恰好与全局深度序一致，所以这一项此处为零）；
3. **$3\sigma$ 包围盒是 tile 对齐的** —— tile 版对该 tile 里的**每个**像素都求值，
   哪怕那个像素在高斯的精确包围盒之外。

所以暴力版必须用**同一个判定集合**才能逐位比对：
它独立地对每个像素算出「该像素所在的 tile 是否落在这个高斯的 tile 区间内」。

In [ ]:
def render_brute(G, T_min=0.0, alpha_min=0.0, bg=(0.05, 0.05, 0.08),
                 step=8, use_tile_box=True):
    '''逐像素暴力：每个像素独立遍历全部高斯（全局深度序）、独立判定包含、合成。
    use_tile_box=True 时用与 tile 版**相同**的判据（该像素所在 tile 是否在高斯的 tile 区间内）；
    False 时不做任何包围盒截断（用来量出这个截断本身的影响）。
    step 只算一个稀疏的像素子集，否则太慢。'''
    order = np.argsort(G['depth'], kind='stable')          # 全局深度序
    img = np.zeros((H_IMG//step, W_IMG//step, 3))
    Tm = np.ones((H_IMG//step, W_IMG//step))
    # 每个高斯的 tile 区间（独立算，不用 bin_gaussians 的结果）
    box = [tile_range(G['uv'][i], G['radius'][i]) for i in range(len(G['uv']))]
    for iy, y in enumerate(range(0, H_IMG, step)):
        for ix, x in enumerate(range(0, W_IMG, step)):
            my_tx, my_ty = x//TS, y//TS
            T = 1.0; acc = np.zeros(3)
            for gi in order:
                if T < T_min:
                    break
                if use_tile_box:
                    bx0, bx1, by0, by1 = box[gi]
                    if not (bx0 <= my_tx <= bx1 and by0 <= my_ty <= by1):
                        continue
                dx = x - G['uv'][gi, 0]; dy = y - G['uv'][gi, 1]
                a_, b_, c_ = G['conic'][gi]
                al = G['alpha'][gi]*np.exp(-0.5*(a_*dx*dx + 2*b_*dx*dy + c_*dy*dy))
                if al < alpha_min:
                    continue
                acc += T*al*G['color'][gi]; T *= (1-al)
            img[iy, ix] = acc; Tm[iy, ix] = T
    return img + Tm[..., None]*np.array(bg), Tm

print('关掉提前终止（T_min=0, alpha_min=0），用同一判定集合比对稀疏像素子集...')
t0 = time.time()
img_b, Tm_b = render_brute(G, step=16, use_tile_box=True)
print(f'暴力版 {time.time()-t0:.1f} s（{(H_IMG//16)*(W_IMG//16)} 个像素）')

img_t, Tmap_t, _ = render_tiles(G, gidx, tidx, T_min=0.0, alpha_min=0.0)
sub = img_t[::16, ::16]
sub_T = Tmap_t[::16, ::16]
d_img = np.abs(sub - img_b).max()
d_T = np.abs(sub_T - Tm_b).max()
print(f'\n图像最大绝对差 {d_img:.3e}')
print(f'透射率最大绝对差 {d_T:.3e}')
assert d_img < 1e-12, f'关掉近似后必须逐位相等，实测 {d_img:.3e}'
assert d_T < 1e-12
print('✓ **严格相等到机器精度** —— 说明 tile 版的分箱、键、tile 区间划分全部正确')
print('  （暴力版独立地对每个像素判定包含关系，没有用 bin_gaussians 的任何输出，')
print('   所以这确实是一次交叉验证，而不是把同一段逻辑跑两遍）')

# 那么第三个近似（tile 对齐的 3σ 包围盒）本身有多大影响？
print('\n第三个近似：tile 对齐的 3σ 包围盒。把它整体去掉试试...')
t0 = time.time()
img_nb, Tm_nb = render_brute(G, step=32, use_tile_box=False)
print(f'  无任何包围盒的暴力版 {time.time()-t0:.1f} s')
img_wb, Tm_wb = render_brute(G, step=32, use_tile_box=True)
d_box = np.abs(img_wb - img_nb).max()
print(f'  有盒 vs 无盒 的最大差 {d_box:.3e}（相当于 {d_box*255:.2f} 个 8bit 色阶）')
assert d_box > 0, '包围盒确实是一个近似，不该恰好为零'
assert d_box < 0.02, f'但它应当很小，实测 {d_box:.3e}'
print('  ✓ 3σ 截断的影响是 %.2f 个色阶量级 —— 小，但**不为零**。' % (d_box*255))
print('    而模块 02 量过：3σ 处的 α 衰减因子 0.0111，对 α≈0.9 的高斯仍有 2.8 个色阶，')
print('    所以这个量级完全对得上，不是数值噪声。')

# 能量守恒：Σw + T_final = 1
w_sum = (img_t - Tmap_t[..., None]*np.array([0.05,0.05,0.08]))
# 用一个全白颜色重渲一遍来直接测权重和
G_white = dict(G); G_white['color'] = np.ones_like(G['color'])
img_w, Tm_w, _ = render_tiles(G_white, gidx, tidx, T_min=0.0, alpha_min=0.0, bg=(0,0,0))
resid = np.abs(img_w[..., 0] + Tm_w - 1.0).max()
print(f'\n能量守恒 |Σw + T_final - 1| 的最大值 {resid:.3e}')
assert resid < 1e-12, '必须逐像素守恒'
print('✓ 逐像素守恒到机器精度')

# 而开了提前终止后，守恒被破坏，破坏量恰好被 T_min 约束
for tm in [1e-2, 1e-4]:
    img_w2, Tm_w2, _ = render_tiles(G_white, gidx, tidx, T_min=tm, alpha_min=0.0, bg=(0,0,0))
    r2 = np.abs(img_w2[..., 0] + Tm_w2 - 1.0).max()
    print(f'  T_min={tm:.0e}: 守恒残差最大 {r2:.3e}  (上界 {tm:.0e})')
    assert r2 <= tm + 1e-12, f'残差必须被 T_min 约束'
print('✓ 提前终止破坏守恒，而破坏量严格被 T_min 约束 —— 这也是一个可验证的量')

## 6 · 顺序不变性与每帧的操作计数

In [ ]:
# ① 打乱输入数组顺序，输出必须不变（因为会重新排序）
perm = np.random.default_rng(7).permutation(n_g)
G_p = {k: v[perm] for k, v in G.items()}
inv = np.empty(n_g, int); inv[perm] = np.arange(n_g)
gidx_p, tidx_p = bin_gaussians(G_p)
img_p, _, _ = render_tiles(G_p, gidx_p, tidx_p, T_min=0.0, alpha_min=0.0)
d = np.abs(img_p - img_t).max()
print(f'打乱输入数组顺序后，图像最大差 {d:.3e}')
assert d < 1e-12, f'必须完全不变，实测 {d:.3e}（说明排序键有相等项）'
print('✓ 完全不变 —— 说明 32 位深度键下没有影响结果的相等项')

# ② 每帧的操作计数（换算到 1920x1080 / 50 万高斯）
print('\n每帧操作计数（1920x1080, 50 万高斯, 半径 8 px）：')
NG_HD, R_HD = 500_000, 8
NT_HD, NPIX_HD = 8160, 1920*1080
per_hd = (2*R_HD/16+1)**2
P_HD = NG_HD*per_hd
bytes_each = 4*2 + 4*3 + 4*1 + 4*3          # uv + conic + alpha + rgb
print(f'  ① 预处理            {NG_HD/1e6:6.2f} M 次')
print(f'  ② 分箱写条目        {P_HD/1e6:6.2f} M 次')
print(f'  ③ 基数排序(8位/遍)  {P_HD/1e6:6.2f} M × 8 = {P_HD*8/1e6:.1f} M 次扫描')
print(f'  ④ 载入条目          {P_HD/1e6:6.2f} M 个 ≈ {P_HD*bytes_each/1e6:.0f} MB 访存')
print(f'  ⑤ 渲染(不终止)      {NPIX_HD*P_HD/NT_HD/1e6:6.0f} M 次高斯-像素求值')
print(f'     渲染(每像素 19)  {NPIX_HD*19/1e6:6.0f} M 次  -> 省 {1-19/(P_HD/NT_HD):.1%}')
print(f'\n  P/高斯数 = {P_HD/NG_HD:.1f}   渲染求值/P = {NPIX_HD*19/P_HD:.1f}'
      f'（不终止时 {NPIX_HD*(P_HD/NT_HD)/P_HD:.1f}）')
assert abs(P_HD/NG_HD - 4.0) < 1e-9

# ③ 负载不均衡：本场景 + 一个 1920x1080 的场景
print('\n负载不均衡（本场景 640x480, 2940 个高斯）：')
print(f'  每 tile 列表长: 均值 {counts.mean():.1f}  中位 {np.median(counts):.0f}'
      f'  P95 {np.percentile(counts,95):.0f}  最大 {counts.max()}')
print(f'  最大/均值 = {counts.max()/counts.mean():.2f}×   空 tile {(counts==0).mean():.1%}')

# 换一个 1920x1080、偏向画面下方（地面与物体在下、天空在上）的分布
rng_hd = np.random.default_rng(1)
NG_L = 200_000
nx_hd, ny_hd = math.ceil(1920/16), math.ceil(1080/16)
gx = rng_hd.uniform(0, 1920, NG_L)
gy = 1080*(1 - rng_hd.beta(1.6, 4.0, NG_L))          # 偏向下方
gr = np.abs(rng_hd.lognormal(np.log(6), 0.8, NG_L)) + 1
cnt_hd = np.zeros((ny_hd, nx_hd), np.int64)
for x, y, r in zip(gx, gy, gr):
    x0 = max(0, int((x-r)//16)); x1 = min(nx_hd-1, int((x+r)//16))
    y0 = max(0, int((y-r)//16)); y1 = min(ny_hd-1, int((y+r)//16))
    cnt_hd[y0:y1+1, x0:x1+1] += 1
ch = cnt_hd.ravel()
print(f'\n1920x1080, {NG_L} 个高斯, 半径中位 6 px, 分布偏向画面下方：')
print(f'  每 tile: 均值 {ch.mean():.1f}  中位 {np.median(ch):.0f}'
      f'  P95 {np.percentile(ch,95):.0f}  最大 {ch.max()}  最小 {ch.min()}')
print(f'  最大/均值 = {ch.max()/ch.mean():.2f}×   空 tile {(ch==0).mean():.1%}')
assert ch.max()/ch.mean() > 2.0, '不均衡比应大于 2'
assert counts.max()/counts.mean() > ch.max()/ch.mean(), \
    '本场景（有近处大物体）应比这个平滑分布更不均衡'
print(f'\n✓ 两个场景的不均衡比分别是 {counts.max()/counts.mean():.2f}× 与 {ch.max()/ch.mean():.2f}×。')
print('  本场景更糟，因为它有一个近处的大物体压住十几个 tile ——')
print('  而这正是真实场景的常态，平滑的合成分布反而低估了不均衡')

# ④ tile 尺寸的取舍
print('\ntile 尺寸的取舍（同一场景）：')
print('  tile     tiles      P          每 tile 列表长')
base = None
for ts in [8, 16, 32]:
    nt = math.ceil(1920/ts)*math.ceil(1080/ts)
    pr = (2*R_HD/ts+1)**2; pp = NG_HD*pr
    if base is None:
        base = (pp, pp/nt)
    print(f'  {ts:2d}x{ts:<2d}  {nt:7d}   {pp/1e6:6.2f}M ({pp/base[0]:5.2f}×)'
          f'   {pp/nt:8.1f} ({(pp/nt)/base[1]:5.2f}×)')
_p8 = NG_HD*(2*R_HD/8+1)**2; _n8 = math.ceil(1920/8)*math.ceil(1080/8)
_p32 = NG_HD*(2*R_HD/32+1)**2; _n32 = math.ceil(1920/32)*math.ceil(1080/32)
assert _p8 > P_HD and (_p8/_n8) < (P_HD/NT_HD), '小 tile：P 变大但列表变短'
assert _p32 < P_HD and (_p32/_n32) > (P_HD/NT_HD), '大 tile：P 变小但列表变长'
print('\n✓ 改 tile 尺寸是把成本在「排序」与「渲染」之间搬来搬去，两项方向相反；')
print('  而改高斯数或覆盖半径是**两项一起涨** —— 所以只有剪枝能真正降成本（模块 04）')

---
## ✏️ 练习

四道题各自独立。先写 TODO，再跑下一格的自测。

### ✏️ 练习 1 · 包围盒到 tile 区间

实现 `my_tile_range(uv, radius, nx, ny, ts)`，返回 `(x0, x1, y0, y1)` —— **闭区间**，
且已裁剪到 `[0, nx-1] × [0, ny-1]`。

In [ ]:
def my_tile_range(uv, radius, nx=NX, ny=NY, ts=TS):
    '''返回覆盖的 tile 闭区间 (x0, x1, y0, y1)，已裁剪。'''
    # TODO: x0 = clip(floor((uv[0]-radius)/ts), 0, nx-1)
    #       x1 = clip(floor((uv[0]+radius)/ts), 0, nx-1)   <- 用 +radius，且是 floor
    #       y 同理。返回 int。
    raise NotImplementedError

In [ ]:
# ---- 自测 1 ----
# ① 完全在中心的小高斯只覆盖 1 个 tile
_r = my_tile_range(np.array([8.0, 8.0]), 1.0)
assert _r == (0, 0, 0, 0), f'应只覆盖 tile (0,0)，实得 {_r}'
# ② 正好跨越 tile 边界
_r = my_tile_range(np.array([16.0, 16.0]), 1.0)
assert _r == (0, 1, 0, 1), f'跨边界应覆盖 2x2，实得 {_r}'
# ③ 屏幕外的负坐标必须被裁到 0
_r = my_tile_range(np.array([-100.0, -100.0]), 5.0)
assert _r == (0, 0, 0, 0), f'负坐标应裁到 0，实得 {_r}'
# ④ 超出右下角必须被裁到 nx-1 / ny-1
_r = my_tile_range(np.array([W_IMG+500.0, H_IMG+500.0]), 5.0)
assert _r == (NX-1, NX-1, NY-1, NY-1), f'应裁到右下角，实得 {_r}'
# ⑤ 一个覆盖整屏的巨大高斯
_r = my_tile_range(np.array([CX, CY]), 10_000.0)
assert _r == (0, NX-1, 0, NY-1), f'应覆盖全部 tile，实得 {_r}'
# ⑥ 与参考实现在随机输入上一致
_rg = np.random.default_rng(2)
for _ in range(3000):
    _uv = _rg.uniform(-50, W_IMG+50, 2)
    _rad = _rg.uniform(0.5, 60)
    assert my_tile_range(_uv, _rad) == tuple(int(v) for v in tile_range(_uv, _rad)), \
        f'与参考不符: uv={_uv} r={_rad}'
# ⑦ tile 数与 (2r/ts+1)² 的量级一致
_cnt = []
for _ in range(2000):
    _uv = _rg.uniform(100, W_IMG-100, 2); _rad = _rg.uniform(2, 30)
    _x0, _x1, _y0, _y1 = my_tile_range(_uv, _rad)
    _cnt.append(((_x1-_x0+1)*(_y1-_y0+1), (2*_rad/TS+1)**2))
_cnt = np.array(_cnt)
_ratio = _cnt[:, 0].mean()/_cnt[:, 1].mean()
assert 0.85 < _ratio < 1.35, f'实测/理论 = {_ratio:.3f}，应接近 1'
print(f'✓ 练习 1 通过：7 类边界情形 + 3000 组随机输入；'
      f'实测 tile 数 / (2r/16+1)² = {_ratio:.3f}')

### 📖 参考答案 1

In [ ]:
def my_tile_range(uv, radius, nx=NX, ny=NY, ts=TS):
    x0 = int(np.clip(np.floor((uv[0]-radius)/ts), 0, nx-1))
    x1 = int(np.clip(np.floor((uv[0]+radius)/ts), 0, nx-1))
    y0 = int(np.clip(np.floor((uv[1]-radius)/ts), 0, ny-1))
    y1 = int(np.clip(np.floor((uv[1]+radius)/ts), 0, ny-1))
    return x0, x1, y0, y1

print('参考答案 1 已定义')
print('两个容易错的地方：')
print('  ① 上界要用 floor((uv+r)/ts) 而不是 ceil。因为 tile 编号是「像素//ts」，')
print('     所以 x=16 属于 tile 1，而 ceil(16/16)=1 恰好对，但 x=15 时 ceil 给 1（错，应是 0）。')
print('  ② 裁剪必须在 floor **之后**。先裁 uv 再 floor 会把「部分在屏幕外」的高斯')
print('     的覆盖范围算小，导致边缘的 tile 漏掉它 —— 表现为画面四边有一条缺失。')

### ✏️ 练习 2 · 64 位排序键

实现 `my_keys(tile_ids, depths)`：返回 `uint64` 键，高 32 位是 `tile_id`，
低 32 位是深度的**单调**编码（用 float32 位重解释，不要线性量化）。

In [ ]:
def my_keys(tile_ids, depths):
    '''返回 uint64 键数组。'''
    # TODO: 1) t = tile_ids 转 np.uint64
    #       2) d = np.asarray(depths, np.float32).view(np.uint32).astype(np.uint64)
    #          （先断言 depths 全为正 —— 这个技巧只对正数成立）
    #       3) 返回 (t << np.uint64(32)) | d
    raise NotImplementedError

In [ ]:
# ---- 自测 2 ----
_t = np.array([5, 5, 5, 2, 2, 9], np.uint64)
_d = np.array([3.0, 1.0, 2.0, 8.0, 0.5, 4.0], np.float32)
_k = my_keys(_t, _d)
assert _k.dtype == np.uint64, f'必须是 uint64，实得 {_k.dtype}'
_o = np.argsort(_k, kind='stable')
assert list(_t[_o]) == [2, 2, 5, 5, 5, 9], f'tile 必须成组升序，实得 {list(_t[_o])}'
assert list(_d[_o]) == [0.5, 8.0, 1.0, 2.0, 3.0, 4.0], \
    f'组内深度必须升序，实得 {list(_d[_o])}'

# 高 32 位必须精确恢复 tile_id
assert list((_k >> np.uint64(32)).astype(np.int64)) == list(_t.astype(np.int64))
# 低 32 位必须精确恢复深度
_lo = (_k & np.uint64(0xFFFFFFFF)).astype(np.uint32)
assert np.array_equal(_lo.view(np.float32), _d), '低 32 位必须无损恢复 float32 深度'

# 大规模：与参考实现一致，且排序结果正确
_kb = my_keys(tidx, G['depth'][gidx])
assert np.array_equal(_kb, make_keys(tidx, G['depth'][gidx])), '与参考实现不符'
_ob = np.argsort(_kb, kind='stable')
_st, _sd = tidx[_ob], G['depth'][gidx][_ob]
assert np.all(np.diff(_st.astype(np.int64)) >= 0)
_sm = np.diff(_st.astype(np.int64)) == 0
assert np.all(np.diff(_sd)[_sm] >= -1e-7)

# 打乱输入后排序结果必须一致（没有影响结果的相等键）
_pm = np.random.default_rng(3).permutation(len(_kb))
_oa = G['depth'][gidx][np.argsort(_kb, kind='stable')]
_obp = G['depth'][gidx][_pm[np.argsort(_kb[_pm], kind='stable')]]
assert np.array_equal(_oa, _obp), '打乱输入不该改变排序结果'

# 负深度必须被拒绝
try:
    my_keys(np.array([0], np.uint64), np.array([-1.0], np.float32))
    raise SystemExit('负深度必须被拒绝（位重解释对负数不单调）')
except AssertionError:
    pass
print(f'✓ 练习 2 通过：{len(_kb)} 个键排序正确；高/低 32 位可无损恢复；'
      f'打乱输入不变；负深度被拒绝')

### 📖 参考答案 2

In [ ]:
def my_keys(tile_ids, depths):
    d32 = np.asarray(depths, np.float32)
    assert np.all(d32 > 0), '位重解释只对正 float32 单调（依赖近平面剔除 z>0.2）'
    t = np.asarray(tile_ids, np.uint64)
    d = d32.view(np.uint32).astype(np.uint64)
    return (t << np.uint64(32)) | d

print('参考答案 2 已定义')
print('要点一：为什么用位重解释而不是线性量化 —— 量化到 16 位时，')
print('       0.5~100 m 范围内有 68.75% 的相邻对键相等，顺序变成未定义。')
print('要点二：这个技巧依赖「深度全为正」，而那是**近平面剔除**保证的。')
print('       两个看起来无关的实现细节其实是耦合的：去掉 z>0.2 的剔除，排序就会错。')
print('       （IEEE 754：正 float 的符号位为 0，指数在高位、尾数在低位，')
print('        所以位模式作为无符号整数与数值同序。负数会反序。）')

### ✏️ 练习 3 · 一个 tile 的渲染（含提前终止）

实现 `my_render_tile(G, g_list, y0, y1, x0, x1, T_min, alpha_min)`，
返回 `(acc, T, cnt)`：该 tile 内每个像素的累积颜色、剩余透射率、实际处理的高斯数。
`g_list` 是**已按深度排好序**的高斯索引。

In [ ]:
def my_render_tile(G, g_list, y0, y1, x0, x1, T_min=1e-4, alpha_min=1.0/255):
    '''返回 (acc (h,w,3), T (h,w), cnt (h,w))。'''
    # TODO: yy, xx = np.mgrid[y0:y1, x0:x1]；T=ones, acc=zeros, cnt=zeros(int)
    #  按 g_list 顺序遍历 gi：
    #    live = T >= T_min；若 not live.any(): break
    #    dx = xx - uv[gi,0]; dy = yy - uv[gi,1]
    #    a,b,c = conic[gi]；m2 = a*dx² + 2b*dx*dy + c*dy²
    #    al = alpha[gi]*exp(-0.5*m2)
    #    use = live & (al >= alpha_min)；若 not use.any(): continue
    #    acc += where(use, T*al, 0)[...,None]*color[gi]
    #    T = where(use, T*(1-al), T)；cnt += use
    raise NotImplementedError

In [ ]:
# ---- 自测 3 ----
_keys3 = make_keys(tidx, G['depth'][gidx])
_o3 = np.argsort(_keys3, kind='stable')
_gs, _ts = gidx[_o3], tidx[_o3]
_s3 = np.searchsorted(_ts, np.arange(NTILE), 'left')
_e3 = np.searchsorted(_ts, np.arange(NTILE), 'right')
# 挑几个非空的 tile
_busy = np.argsort(_e3-_s3)[::-1][:6]

for _tid in _busy:
    _ty, _tx = divmod(int(_tid), NX)
    _y0, _y1 = _ty*TS, min((_ty+1)*TS, H_IMG)
    _x0, _x1 = _tx*TS, min((_tx+1)*TS, W_IMG)
    _gl = _gs[_s3[_tid]:_e3[_tid]]

    # ① 关掉两个近似时，必须与完整合成逐位相等
    _acc, _T, _cnt = my_render_tile(G, _gl, _y0, _y1, _x0, _x1, 0.0, 0.0)
    assert _acc.shape == (_y1-_y0, _x1-_x0, 3) and _T.shape == _acc.shape[:2]
    assert np.all(_cnt == len(_gl)), '关掉近似时每个像素都该处理全部高斯'
    # 能量守恒（用全白颜色）
    _Gw = dict(G); _Gw['color'] = np.ones_like(G['color'])
    _accw, _Tw, _ = my_render_tile(_Gw, _gl, _y0, _y1, _x0, _x1, 0.0, 0.0)
    assert np.abs(_accw[..., 0] + _Tw - 1.0).max() < 1e-12, '必须逐像素守恒'

    # ② 提前终止：误差被 T_min 约束，且处理的高斯数不增加
    for _tm in [1e-2, 1e-4]:
        _a2, _T2, _c2 = my_render_tile(G, _gl, _y0, _y1, _x0, _x1, _tm, 0.0)
        assert np.abs(_a2 - _acc).max() <= _tm + 1e-12, \
            f'T_min={_tm} 时误差 {np.abs(_a2-_acc).max():.3e} 超上界'
        assert np.all(_c2 <= _cnt), '提前终止不该增加处理量'

    # ③ T 单调不增、落在 (0,1]
    assert np.all(_T > 0) and np.all(_T <= 1.0 + 1e-15)
    # ④ 与本 notebook 的整帧实现在该 tile 上一致
    assert np.abs(_acc - (img_t - Tmap_t[..., None]*np.array([0.05,0.05,0.08]))
                  [_y0:_y1, _x0:_x1]).max() < 1e-12, '与整帧实现不一致'

# ⑤ 空列表
_ae, _Te, _ce = my_render_tile(G, np.array([], int), 0, 16, 0, 16, 1e-4, 0.0)
assert np.allclose(_ae, 0) and np.allclose(_Te, 1.0) and np.all(_ce == 0)
print(f'✓ 练习 3 通过：6 个最繁忙的 tile（列表长 {int(_e3[_busy[0]]-_s3[_busy[0]])} '
      f'~ {int(_e3[_busy[-1]]-_s3[_busy[-1]])}）；守恒、误差上界、单调性、空列表全部通过')

### 📖 参考答案 3

In [ ]:
def my_render_tile(G, g_list, y0, y1, x0, x1, T_min=1e-4, alpha_min=1.0/255):
    yy, xx = np.mgrid[y0:y1, x0:x1]
    T = np.ones(yy.shape)
    acc = np.zeros(yy.shape + (3,))
    cnt = np.zeros(yy.shape, int)
    for gi in g_list:
        live = T >= T_min
        if not live.any():
            break
        dx = xx - G['uv'][gi, 0]; dy = yy - G['uv'][gi, 1]
        a_, b_, c_ = G['conic'][gi]
        m2 = a_*dx*dx + 2*b_*dx*dy + c_*dy*dy
        al = G['alpha'][gi] * np.exp(-0.5*m2)
        use = live & (al >= alpha_min)
        if not use.any():
            continue
        acc += np.where(use, T*al, 0.0)[..., None] * G['color'][gi]
        T = np.where(use, T*(1-al), T)
        cnt += use
    return acc, T, cnt

print('参考答案 3 已定义')
print('要点一：`live = T >= T_min` 要在**处理之前**判，否则误差上界不再是 T_min。')
print('要点二：`break` 与 `continue` 的区别是本函数最容易错的地方 ——')
print('       T < T_min 时 break（后面更远的高斯只会更被遮挡）；')
print('       α < alpha_min 时 continue（这个高斯太淡，但后面的可能不淡）。')
print('       写反了：break 太早会丢内容，continue 代替 break 会白算 92% 的工作。')
print('要点三：GPU 上的 live 是 per-thread 的 break，而 `not live.any()` 对应')
print('       __syncthreads_count(done)==256 那个 block 级退出 —— 省访存的那一层。')

### ✏️ 练习 4 · 深度量化位数的碰撞率

实现 `my_collision(depths, bits)`：把 `depths` 线性量化到 `bits` 位后，
返回**排序后相邻键相等**的比例（即顺序未定义的相邻对占比）。

In [ ]:
def my_collision(depths, bits):
    '''线性量化到 bits 位后，排序后相邻相等的比例。'''
    # TODO: lo, hi = depths.min(), depths.max()
    #       q = ((depths-lo)/(hi-lo)*(2**bits-1)).astype(np.uint64)
    #       返回 (np.diff(np.sort(q)) == 0).mean()
    raise NotImplementedError

In [ ]:
# ---- 自测 4 ----
_dw = np.random.default_rng(0).uniform(0.5, 100.0, 200_000)
_r = {b: my_collision(_dw, b) for b in [16, 20, 24, 32]}

for _b, _v in _r.items():
    assert 0.0 <= _v <= 1.0, f'{_b} 位: {_v} 不是比例'
# 位数越多碰撞越少（严格单调）
assert _r[16] > _r[20] > _r[24] > _r[32], f'必须单调: {_r}'
# 具体量级
assert 0.60 < _r[16] < 0.75, f'16 位应在 60~75%，实测 {_r[16]:.2%}'
assert _r[32] < 1e-3, f'32 位应 <0.1%，实测 {_r[32]:.4%}'
# 与参考实现一致
for _b in [16, 24, 32]:
    assert abs(my_collision(_dw, _b) - collision_rate(_dw, _b)) < 1e-12

# 理论核对：n 个值均匀落进 m 个桶时，碰撞率 ≈ 1 - m/n·(1-exp(-n/m)) 的近似不好用，
# 直接用「非空桶数」核对：相邻相等的对数 = n - 非空桶数
for _b in [16, 20]:
    _lo, _hi = _dw.min(), _dw.max()
    _q = ((_dw-_lo)/(_hi-_lo)*(2**_b-1)).astype(np.uint64)
    _n_distinct = len(np.unique(_q))
    _expect = (len(_dw) - _n_distinct) / (len(_dw) - 1)
    assert abs(my_collision(_dw, _b) - _expect) < 1e-9, \
        f'{_b} 位: 碰撞率应等于 (n - 不同值个数)/(n-1)'

# 极端：1 位时几乎全部碰撞
assert my_collision(_dw, 1) > 0.99
# 深度全相同时，任何位数都是 100% 碰撞
assert my_collision(np.full(1000, 7.0), 32) == 1.0
print(f'✓ 练习 4 通过：16 位 {_r[16]:.2%} / 20 位 {_r[20]:.2%} / '
      f'24 位 {_r[24]:.2%} / 32 位 {_r[32]:.4%}；与「(n-不同值数)/(n-1)」恒等式一致')

### 📖 参考答案 4

In [ ]:
def my_collision(depths, bits):
    d = np.asarray(depths, float)
    lo, hi = d.min(), d.max()
    if hi == lo:
        return 1.0
    q = ((d - lo)/(hi - lo)*(2**int(bits) - 1)).astype(np.uint64)
    return float((np.diff(np.sort(q)) == 0).mean())

print('参考答案 4 已定义')
print('要点：碰撞率有一个精确的恒等式 —— (n - 不同值个数)/(n-1)。')
print('     所以它其实是在数「量化后还剩多少个不同的深度」，')
print('     而这正是「排序还能区分多少层」。16 位下 20 万个深度只剩 6 万多个不同值。')
print()
print('工程结论：不要线性量化，直接位重解释 float32（练习 2）。')
print('        而这条结论只能通过「打乱输入顺序，输出应不变」这条测试发现 ——')
print('        只看 PSNR 的话，它表现为莫名的 0.5 dB 损失。')

---
## 🧪 真实工程胶囊

```python
# ---- 官方实现：分箱与排序（cuda_rasterizer/rasterizer_impl.cu）----
# 1) 每个高斯算出 tile 区间，用前缀和确定它在全局数组里的写入位置
#    getRect(p_screen, my_radius, rect_min, rect_max, grid);
#    cub::DeviceScan::InclusiveSum(..., tiles_touched, point_offsets, P);
#    num_rendered = point_offsets[P-1];              // ← 练习 1 的 P
#
# 2) 生成 64 位键：高 32 位 tile_id，低 32 位深度的位重解释
#    duplicateWithKeys<<<...>>>(...):
#        uint64_t key = y * grid.x + x;              // tile_id
#        key <<= 32;
#        key |= *((uint32_t*)&depths[idx]);          // ← 练习 2 的位重解释
#
# 3) 一次基数排序
#    cub::DeviceRadixSort::SortPairs(
#        list_sorting_space, sorting_size,
#        point_list_keys_unsorted, point_list_keys,
#        point_list_unsorted, point_list, num_rendered, 0, 32 + bit);
#    // 注意最后两个参数：只排低 (32+bit) 位，bit = ceil(log2(tile 数))
#    // 8160 个 tile -> bit=13 -> 只排 45 位，省下 19 位的扫描
#
# 4) 找每个 tile 的区间
#    identifyTileRanges<<<...>>>(num_rendered, point_list_keys, imgState.ranges);

# ---- gsplat 的对应接口（更容易读，也能单独调用每一步）----
from gsplat import fully_fused_projection, isect_tiles, isect_offset_encode, rasterize_to_pixels
radii, means2d, depths, conics, compensations = fully_fused_projection(
    means, None, quats, scales, viewmats, Ks, width, height)
tiles_per_gauss, isect_ids, flatten_ids = isect_tiles(
    means2d, radii, depths, tile_size=16,
    tile_width=math.ceil(W/16), tile_height=math.ceil(H/16))
isect_offsets = isect_offset_encode(isect_ids, C, tile_width, tile_height)
colors, alphas = rasterize_to_pixels(
    means2d, conics, colors, opacities, width, height, 16, isect_offsets, flatten_ids)
# tiles_per_gauss 就是练习 1 算的东西；isect_ids 就是练习 2 的键

# ---- 自己改内核时的五条验证（第 7 节）----
# 1) T_min=0 + tile=1x1 时与逐像素暴力实现**逐位相等**
# 2) 逐个关掉近似做消融，差应降到 0
# 3) 梯度用数值差分核对（注意提前终止让损失不光滑）
# 4) 打乱输入**数组**顺序，输出必须不变   <- 抓排序键的相等项
# 5) Σw + T_final = 1 逐像素成立；开了终止后残差被 T_min 约束
```

**排查清单**

| 症状 | 先查什么 | 依据 |
|---|---|---|
| 画面四边缺一条 | `tile_range` 是先裁剪还是先 floor | 先裁 uv 会把部分出屏的高斯覆盖范围算小 |
| PSNR 莫名低 0.5 dB，图看不出问题 | 深度键的位数；排序是否稳定 | 16 位量化时 68.75% 的相邻对顺序未定义 |
| 相机转动时画面「跳一下」 | popping：tile 内共享顺序 | 两个穿插的椭球在一个 tile 里只有一个顺序 |
| 性能远低于预期 | 有没有超大的高斯 | 半径 100 px 的一个高斯 ≈ 44 个正常高斯的分箱成本 |
| 训练早期极慢、后期突然变快 | α 初值太小 → 无法提前终止 | 官方初值 α=0.1，此时没有 tile 能终止 |
| 改小 tile 反而更慢 | 成本被搬到了排序侧 | 8×8: P ×2.25 而列表长 ×0.57 |
| 反向比前向慢很多 | atomicAdd 冲突 | 8 px 的高斯覆盖 201 个像素 |